# Sesión 3 - Práctica 2: Agentes Comunicándose (EN VIVO)

## Objetivos
- Crear dos agentes que se comunican mediante mensajes
- Usar mensajes FIPA-ACL (performatives)
- Implementar el protocolo REQUEST
- Ver el flujo completo: enviar mensaje, recibir, responder

## Duración estimada
25 minutos

## NOTA IMPORTANTE
Este notebook tiene gaps (huecos) marcados con TODO que completaremos juntos en clase.

## Parte 1: Agente Receptor (Escucha Mensajes)

### GAP 1: Completar la recepción de mensajes

In [1]:
from spade import agent
from spade.behaviour import CyclicBehaviour
from spade.message import Message
import asyncio

class AgenteReceptor(agent.Agent):
    """
    Agente que escucha mensajes y responde.
    """
    
    class ComportamientoEscucha(CyclicBehaviour):
        """
        Behaviour que espera mensajes constantemente.
        """
        
        async def run(self):
            """
            Espera mensajes y los procesa.
            """
            # TODO: Esperar un mensaje con timeout de 10 segundos
            msg = await self.receive(timeout=10)
            
            
            if msg:
                print(f"\n[RECEPTOR] Mensaje recibido!")
                
                # TODO: Imprimir el sender del mensaje
                print(f"  De: {msg.sender}")
                
                
                # TODO: Imprimir el performative del mensaje
                print(f"  Tipo: {msg.get_metadata('performative')}")
                
                
                print(f"  Contenido: {msg.body}")
                
                # Procesar según el tipo de mensaje
                performative = msg.get_metadata('performative')
                
                if performative == 'request':
                    await self.procesar_peticion(msg)
                elif performative == 'query-if':
                    await self.procesar_pregunta(msg)
                else:
                    print(f"  Tipo de mensaje no reconocido: {performative}")
        
        async def procesar_peticion(self, msg):
            """
            Procesa una petición (REQUEST) y responde.
            """
            contenido = msg.body.lower()
            
            if 'hora' in contenido:
                # Aceptar la petición
                print("[RECEPTOR] Puedo responder eso. Enviando AGREE...")
                
                # TODO: Crear un mensaje de respuesta al sender
                respuesta = Message(to=str(msg.sender))
                
                
                # TODO: Establecer el performative como 'agree'
                respuesta.set_metadata('performative', 'agree')
                
                
                respuesta.body = "Voy a darte la hora actual"
                
                # TODO: Enviar la respuesta
                await self.send(respuesta)
                
                
                # Hacer la tarea
                await asyncio.sleep(1)
                import datetime
                hora_actual = datetime.datetime.now().strftime("%H:%M:%S")
                
                # Enviar resultado con INFORM
                resultado = Message(to=str(msg.sender))
                resultado.set_metadata('performative', 'inform')
                resultado.body = f"La hora actual es {hora_actual}"
                await self.send(resultado)
                
                print(f"[RECEPTOR] Enviado resultado: {hora_actual}")
            else:
                # Rechazar la petición
                print("[RECEPTOR] No puedo hacer eso. Enviando REFUSE...")
                
                respuesta = Message(to=str(msg.sender))
                respuesta.set_metadata('performative', 'refuse')
                respuesta.body = "No entiendo esa petición"
                await self.send(respuesta)
        
        async def procesar_pregunta(self, msg):
            """
            Procesa una pregunta (QUERY-IF) y responde.
            """
            print("[RECEPTOR] Es una pregunta. Respondiendo...")
            
            respuesta = Message(to=str(msg.sender))
            respuesta.set_metadata('performative', 'inform')
            respuesta.body = "Respuesta a tu pregunta: Si"
            await self.send(respuesta)
    
    async def setup(self):
        print(f"[RECEPTOR] Agente {self.jid} iniciando...")
        comportamiento = self.ComportamientoEscucha()
        self.add_behaviour(comportamiento)
        print("[RECEPTOR] Escuchando mensajes...")

Using slower stringprep, consider compiling the faster cython/libidn one.


## Parte 2: Agente Emisor (Envía Mensajes)

### GAP 2: Completar el envío de mensajes

In [2]:
from spade.behaviour import OneShotBehaviour

class AgenteEmisor(agent.Agent):
    """
    Agente que envía un mensaje y espera respuesta.
    """
    
    class ComportamientoEnvio(OneShotBehaviour):
        """
        Behaviour que envía un mensaje una vez.
        """
        
        async def run(self):
            """
            Envía un mensaje y espera respuesta.
            """
            print("\n[EMISOR] Preparando mensaje...")
            
            # Esperar un poco para que el receptor esté listo
            await asyncio.sleep(2)
            
            # TODO: Crear un mensaje al receptor
            receptor_jid = "agente_receptor_001@anon.jabberfr.org"
            msg = Message(to=receptor_jid)
            
            
            
            # TODO: Establecer el performative como 'request'
            msg.set_metadata('performative', 'request')
            
            
            # TODO: Establecer el contenido del mensaje
            msg.body = "Por favor, dime la hora actual"
            
            
            # TODO: Enviar el mensaje
            await self.send(msg)
            
            
            print(f"[EMISOR] Mensaje REQUEST enviado a {receptor_jid}")
            print(f"  Contenido: {msg.body}")
            
            # Esperar respuesta AGREE o REFUSE
            print("[EMISOR] Esperando respuesta AGREE/REFUSE...")
            respuesta1 = await self.receive(timeout=10)
            
            if respuesta1:
                perf = respuesta1.get_metadata('performative')
                print(f"[EMISOR] Respuesta recibida: {perf}")
                print(f"  Mensaje: {respuesta1.body}")
                
                if perf == 'agree':
                    # El receptor aceptó, esperar resultado
                    print("[EMISOR] Esperando resultado final (INFORM)...")
                    respuesta2 = await self.receive(timeout=10)
                    
                    if respuesta2:
                        print(f"[EMISOR] Resultado recibido!")
                        print(f"  Respuesta: {respuesta2.body}")
                    else:
                        print("[EMISOR] No se recibió resultado (timeout)")
                elif perf == 'refuse':
                    print("[EMISOR] El receptor rechazó la petición.")
            else:
                print("[EMISOR] No se recibió respuesta (timeout)")
            
            # Detener el agente
            await asyncio.sleep(2)
            await self.agent.stop()
    
    async def setup(self):
        print(f"[EMISOR] Agente {self.jid} iniciando...")
        comportamiento = self.ComportamientoEnvio()
        self.add_behaviour(comportamiento)
        print("[EMISOR] Behaviour de envío añadido.")

## Parte 3: Ejecutar Ambos Agentes

In [3]:
async def ejecutar_comunicacion():
    """
    Ejecuta ambos agentes y los deja comunicarse.
    """
    # Crear agentes con diferentes credenciales
    receptor = AgenteReceptor(
        "agente_receptor_001@anon.jabberfr.org",
        "spade_receptor_2025"
    )
    
    emisor = AgenteEmisor(
        "agente_emisor_001@anon.jabberfr.org",
        "spade_emisor_2025"
    )
    
    # Iniciar receptor primero
    await receptor.start()
    print("\n=== Receptor iniciado ===")
    
    # Esperar un poco antes de iniciar emisor
    await asyncio.sleep(2)
    
    # Iniciar emisor
    await emisor.start()
    print("\n=== Emisor iniciado ===")
    
    # Esperar a que el emisor termine (el que se detiene primero)
    while emisor.is_alive():
        await asyncio.sleep(0.5)
    
    print("\n=== Emisor detenido ===")
    
    # Detener receptor
    await receptor.stop()
    print("=== Receptor detenido ===")
    
    print("\n=== Comunicación completada ===")

# Ejecutar
await ejecutar_comunicacion()

[RECEPTOR] Agente agente_receptor_001@anon.jabberfr.org iniciando...
[RECEPTOR] Escuchando mensajes...

=== Receptor iniciado ===
[EMISOR] Agente agente_emisor_001@anon.jabberfr.org iniciando...
[EMISOR] Behaviour de envío añadido.

=== Emisor iniciado ===

[EMISOR] Preparando mensaje...
[EMISOR] Mensaje REQUEST enviado a agente_receptor_001@anon.jabberfr.org
  Contenido: Por favor, dime la hora actual
[EMISOR] Esperando respuesta AGREE/REFUSE...

[RECEPTOR] Mensaje recibido!
  De: agente_emisor_001@anon.jabberfr.org
  Tipo: request
  Contenido: Por favor, dime la hora actual
[RECEPTOR] Puedo responder eso. Enviando AGREE...
[EMISOR] Respuesta recibida: agree
  Mensaje: Voy a darte la hora actual
[EMISOR] Esperando resultado final (INFORM)...
[RECEPTOR] Enviado resultado: 20:47:49
[EMISOR] Resultado recibido!
  Respuesta: La hora actual es 20:47:49

=== Emisor detenido ===
=== Receptor detenido ===

=== Comunicación completada ===


## Parte 4: Protocolo Completo con Múltiples Mensajes

### GAP 3: Completar el agente conversador

In [5]:
class AgenteConversador(agent.Agent):
    """
    Agente que mantiene una conversación más compleja.
    """
    
    class ComportamientoConversacion(OneShotBehaviour):
        async def run(self):
            print("\n[CONVERSADOR] Iniciando conversación...")
            
            receptor_jid = "agente_receptor_001@anon.jabberfr.org"
            
            await asyncio.sleep(2)
            
            # Mensaje 1: REQUEST
            msg1 = Message(to=receptor_jid)
            msg1.set_metadata('performative', 'request')
            msg1.body = "Por favor, dime la hora actual"
            await self.send(msg1)
            print("[CONVERSADOR] Enviado REQUEST: hora actual")
            
            # Esperar respuestas
            resp1 = await self.receive(timeout=10)
            if resp1:
                print(f"[CONVERSADOR] Recibido: {resp1.get_metadata('performative')}")
            
            resp2 = await self.receive(timeout=10)
            if resp2:
                print(f"[CONVERSADOR] Recibido: {resp2.body}")
            
            await asyncio.sleep(1)
            
            # TODO: Crear mensaje 2 con QUERY-IF
            msg2 = Message(to=receptor_jid)
            msg2.set_metadata('performative', 'query-if')
            msg2.body = "Estas disponible?"
            
            
            
            
            # TODO: Enviar mensaje 2
            await self.send(msg2)
            
            
            print("[CONVERSADOR] Enviado QUERY-IF: disponible?")
            
            resp3 = await self.receive(timeout=10)
            if resp3:
                print(f"[CONVERSADOR] Recibido: {resp3.body}")
            
            await asyncio.sleep(2)
            await self.agent.stop()
    
    async def setup(self):
        print("[CONVERSADOR] Iniciando...")
        self.add_behaviour(self.ComportamientoConversacion())

In [6]:
async def ejecutar_conversacion():
    receptor = AgenteReceptor(
        "agente_receptor_001@anon.jabberfr.org",
        "spade_receptor_2025"
    )
    
    conversador = AgenteConversador(
        "agente_conversador_001@anon.jabberfr.org",
        "spade_conversador_2025"
    )
    
    await receptor.start()
    await asyncio.sleep(2)
    await conversador.start()
    
    while conversador.is_alive():
        await asyncio.sleep(0.5)
    
    await receptor.stop()
    print("\n=== Conversación completada ===")

await ejecutar_conversacion()

[RECEPTOR] Agente agente_receptor_001@anon.jabberfr.org iniciando...
[RECEPTOR] Escuchando mensajes...
[CONVERSADOR] Iniciando...

[CONVERSADOR] Iniciando conversación...
[CONVERSADOR] Enviado REQUEST: hora actual

[RECEPTOR] Mensaje recibido!
  De: agente_conversador_001@anon.jabberfr.org
  Tipo: request
  Contenido: Por favor, dime la hora actual
[RECEPTOR] Puedo responder eso. Enviando AGREE...
[CONVERSADOR] Recibido: agree
[RECEPTOR] Enviado resultado: 20:50:34
[CONVERSADOR] Recibido: La hora actual es 20:50:34
[CONVERSADOR] Enviado QUERY-IF: disponible?

[RECEPTOR] Mensaje recibido!
  De: agente_conversador_001@anon.jabberfr.org
  Tipo: query-if
  Contenido: Estas disponible?
[RECEPTOR] Es una pregunta. Respondiendo...
[CONVERSADOR] Recibido: Respuesta a tu pregunta: Si

=== Conversación completada ===


## Resumen

Has aprendido:

1. Crear mensajes FIPA-ACL con performatives:
   - REQUEST: Pedir algo
   - AGREE/REFUSE: Aceptar o rechazar
   - INFORM: Informar resultado
   - QUERY-IF: Preguntar

2. Protocolo REQUEST completo:
   - Emisor envía REQUEST
   - Receptor responde AGREE o REFUSE
   - Si AGREE, receptor ejecuta y envía INFORM con resultado

3. Enviar mensajes: await self.send(msg)
4. Recibir mensajes: msg = await self.receive(timeout=X)
5. Múltiples agentes pueden comunicarse simultáneamente

### Siguiente paso

Ahora puedes empezar a diseñar tu sistema multiagente para la Actividad 1!